# Chapter 13: A Philosophy of Streaming Systems
*From: Designing Data-Intensive Applications*

> "If the highest aim of a captain was to preserve his ship, he would keep it in port forever."
> — St. Thomas Aquinas, *Summa Theologica*

This chapter is a synthesis. It pulls together event logs (Ch 11–12), transactions (Ch 8), consistency/consensus (Ch 10), and derived data (Ch 5) into one **opinionated philosophy**: build applications as **dataflow** over immutable event logs rather than as synchronous RPC against shared mutable state.

## TL;DR

- **The integration problem, not the storage problem, is the hard one.** No single database is right for every access pattern, so real systems always compose several — and keeping them in sync is where distributed transactions fall over.
- **Log-based derived data beats distributed transactions** for heterogeneous integration: an ordered event log plus deterministic, idempotent consumers gives *loose coupling*, fault containment, and evolvability at the cost of timeliness (asynchrony).
- **Unbundle the database.** Think of an organization's dataflow as one giant database whose indexes, materialized views, caches, and replication logs are provided by independent components glued together by Kafka-style logs (Unix pipes) rather than a single integrated engine.
- **End-to-end is the only honest frame for correctness.** TCP checksums, DB transactions, and 2PC each plug one hole. Exactly-once behavior for users requires a client-generated request ID threaded through every hop and enforced at the endpoint (uniqueness constraint, idempotent handler).
- **Separate timeliness from integrity.** Coordination-avoiding designs give up "the moment I commit, everyone sees it" but preserve "no money disappears, no duplicate charge." For many businesses, violations of timeliness are fine (apologize + compensate), violations of integrity are catastrophic.
- **Trust, but verify.** Hardware and software both lie occasionally. Mature systems continually re-derive state from an immutable log and hash-check it end to end — Merkle trees, certificate transparency, and blockchains are the extreme points on the same spectrum.

## Design Principles the Chapter Advocates

Rather than "requirements" in the system-design sense, this chapter argues for a **set of principles** you should bring to any data-intensive application:

**Architectural principles**
- **Pick specialized tools per access pattern.** Accept that you will compose several (OLTP store + search index + warehouse + cache + ML features).
- **Designate a system of record.** Every derived dataset (index, cache, materialized view, ML model) has exactly one source and is rebuilt by a deterministic function.
- **Prefer asynchronous event logs over synchronous RPC** for integration across heterogeneous systems. Logs give ordering, buffering, and loose coupling; RPC couples availability.
- **Application code as a derivation function.** State management lives in logs and managed stores; code is a stateless, deterministic operator (the "Church and state" joke).

**Correctness principles**
- **End-to-end thinking (Saltzer/Reed/Clark).** Correctness properties must be enforced at the endpoints that care about them — lower-level mechanisms only reduce the *probability* of failure.
- **Idempotence via client-generated request IDs.** Thread one unique ID from the end-user device through every layer; enforce uniqueness at the final store.
- **Separate timeliness from integrity.** Integrity (no corruption, no lost events) is non-negotiable; timeliness (linearizable visibility) is a business trade-off.
- **Coordination is a scarce resource.** Reserve synchronous coordination for the few constraints where an apology is unacceptable; handle the rest with compensating transactions.
- **Auditability by default.** Assume hardware and software will occasionally corrupt data; build continuous re-derivation and hash-chaining so you *detect* the corruption.

**Non-functional goals** (from Chapter 2, carried forward): reliability, scalability, maintainability, evolvability. The philosophy exists in service of these — especially **evolvability**, which is why reprocessable immutable logs matter so much.

## Synchronous RPC vs. Asynchronous Dataflow: The Numbers

This chapter is a philosophy, not a capacity-sizing problem. But the core argument — "async log-based dataflow beats synchronous distributed transactions" — is partly a quantitative claim about **availability and tail latency**. The cell below makes that concrete: distributed transactions multiply per-node failures across all shards, and chained synchronous RPC amplifies tail latency exponentially in the number of hops. Async dataflow decouples both.

In [ ]:
# Synchronous RPC / distributed transactions vs. async log-based dataflow
# -----------------------------------------------------------------------
# All numbers illustrative but realistic. We quote DDIA pages 550, 573.
from math import log, sqrt

# ---------- 1. Availability of a multi-shard distributed transaction ----------
# A 2PC / XA transaction across N shards aborts if ANY participant is unavailable
# when the coordinator tries to prepare. If each shard has per-op availability p,
# the whole transaction succeeds with probability p^N. (Ch 8, 13.)
per_node_availability = 0.9999  # "four nines" per shard
print("Distributed transaction success rate (p^N):")
print(f"  {'shards':>7} | {'p(commit)':>11} | {'p(abort)':>10}")
for n in (1, 3, 5, 10, 30, 100):
    p_ok = per_node_availability ** n
    print(f"  {n:>7} | {p_ok:>11.6f} | {1 - p_ok:>10.6f}")
print("  -> a 30-shard 2PC aborts ~0.3% of the time from node noise alone;")
print("     a 100-shard one aborts ~1%. Operators feel this as 'the cluster is flaky'.")

# ---------- 2. Tail-latency amplification of chained synchronous RPC ----------
# If one hop has p50 ~1ms and p99 ~5ms, chaining N independent hops doesn't just
# add p99s -- the *chain's* p99 is dominated by the worst hop, so end-to-end p99
# grows roughly like max of N samples from the hop's latency distribution.
# Under a lognormal tail, E[max of N] grows with sqrt(2 * ln N).
hop_p50_ms = 1.0
hop_p99_ms = 5.0
# sigma such that p99 / p50 = exp(sigma * z_99), z_99 ~ 2.326
sigma = log(hop_p99_ms / hop_p50_ms) / 2.326
print("\nChained synchronous RPC p99 end-to-end (lognormal hop tails):")
print(f"  per-hop p50={hop_p50_ms}ms, p99={hop_p99_ms}ms  ->  sigma={sigma:.3f}")
print(f"  {'hops':>4} | {'chain p50 (ms)':>14} | {'chain p99 (ms)':>14}")
for n in (1, 3, 5, 10, 20):
    chain_p50 = n * hop_p50_ms
    # Worst-of-N tail approximation for the slowest hop in the chain.
    worst_of_n_p99 = hop_p50_ms * pow(2.71828, sigma * (2.326 + sqrt(2 * log(max(n, 2)))))
    chain_p99 = (n - 1) * hop_p50_ms + worst_of_n_p99
    print(f"  {n:>4} | {chain_p50:>14.1f} | {chain_p99:>14.1f}")
print("  -> a 10-hop synchronous call graph's p99 is ~5-10x its p50. Users feel this.")

# ---------- 3. Asynchronous dataflow decouples both ---------------------------
# Log-based dataflow replaces the chain with: producer writes ONE event to a log
# (one local write + one network hop), consumers process independently. The
# producer's latency no longer depends on any downstream consumer, and a slow or
# failed consumer just widens its own lag -- it does not fail the write.
producer_write_ms = 2.0                     # local append to log broker
log_replication_ms = 5.0                    # quorum ack on broker
producer_end_to_end_ms = producer_write_ms + log_replication_ms
print("\nAsync dataflow producer-visible latency (independent of downstream):")
print(f"  write + log quorum ack  = {producer_end_to_end_ms:.1f} ms")
print(f"  downstream consumer lag = bounded by consumer throughput, not coupled")
print(f"  producer availability   = broker availability, NOT product of N shards")

# ---------- 4. The trade: timeliness ------------------------------------------
# The cost of async is staleness at the derived view. A consumer processing
# events_per_sec with incoming rate r events/sec has steady-state lag ~ queue/r.
incoming_events_per_sec = 50_000
consumer_throughput     = 60_000
utilization = incoming_events_per_sec / consumer_throughput
# M/M/1-ish expected lag in seconds while caught up (illustrative)
expected_lag_sec = (utilization / (1 - utilization)) / consumer_throughput
print(f"\nDerived-view staleness at rho={utilization:.2f}: "
      f"~{expected_lag_sec*1000:.2f} ms of lag.")
print("Integrity is preserved (every event eventually lands); only timeliness slips.")

## High-Level Design: The Unbundled Database

The architecture the chapter advocates is a dataflow graph with the event log at its center. The system of record writes once; everything else is a *derived* view computed by a stream processor and eventually pushed all the way to the end-user device.

```mermaid
flowchart LR
    User[End-user device<br/>React/Elm UI]
    App[App server<br/>stateless]
    Log[(Event log<br/>Kafka-style<br/>sharded, ordered)]
    SP1[Stream processor:<br/>search indexer]
    SP2[Stream processor:<br/>materialized view]
    SP3[Stream processor:<br/>ML features]
    SP4[Stream processor:<br/>cache rebuilder]
    Search[(Search index)]
    MV[(Materialized<br/>view)]
    Model[(ML model /<br/>feature store)]
    Cache[(Cache)]
    WH[(Data warehouse)]
    Push[WebSocket /<br/>server-sent events]

    User -- "write + request_id" --> App
    App -- "append event" --> Log
    Log --> SP1 --> Search
    Log --> SP2 --> MV
    Log --> SP3 --> Model
    Log --> SP4 --> Cache
    Log --> WH
    MV --> Push --> User
    Search --> App
    Cache --> App
```

**Read this diagram as:** the log is the system of record. Every other store is a view. Any view can be **dropped and rebuilt** from the log. The write path is eager precomputation; the read path is lazy query; the **derived store is where they meet**, and that boundary can be pushed all the way out to the browser via WebSockets.

## Deep Dive

### 1. Data Integration via Derived Data

Every nontrivial application needs more than one storage engine — OLTP + search index + warehouse + cache + ML features. The historical answer was **distributed transactions (XA / 2PC)**. The chapter argues: don't. Instead, funnel writes through **one ordered event log**, make each derivation **deterministic and idempotent**, and let each downstream system consume independently.

Why logs beat 2PC in practice:

- **Fault containment.** A slow consumer doesn't abort anyone; 2PC does.
- **Heterogeneity.** Logs don't need a shared XA protocol across every tool in the stack.
- **Recoverable evolution.** You can reprocess the log to rebuild a derived store with new logic.
- **Loose coupling.** Producers don't know about consumers; consumers don't block producers.

Price paid: asynchronous views. "Read your writes" is no longer free — you have to either wait on the output stream or build it yourself.

```mermaid
flowchart LR
    SoR[(System of record<br/>OLTP DB)]
    CDC[CDC / event log<br/>total order per shard]
    SoR --> CDC
    CDC --> Idx[(Search index)]
    CDC --> Cache[(Cache)]
    CDC --> WH[(Warehouse)]
    CDC --> ML[(ML pipeline)]
    CDC --> Notif[Notifications]
```

See [[01-data-integration-via-derived-data]].

### 2. Unbundling the Database

Databases already implement indexes, replication, materialized views, triggers, and stored procedures internally. **An organization's dataflow is the same machinery, spread across many teams.**

Two ways to compose heterogeneous storage:

- **Federated databases (unifying reads).** One query engine over many backends — Trino, Postgres foreign data wrappers. Relational tradition: integrated, declarative, elegant semantics, hard implementation.
- **Unbundled databases (unifying writes).** CDC + event logs wire the stores together; each store does one thing well. Unix tradition: small tools, uniform pipe interface, compose with a shell.

Federation is easier (read-only translation); **unbundling is the hard problem** because writes must stay consistent across heterogeneous tools. The answer is the event log.

```mermaid
flowchart TB
    subgraph Federated["Federated DB (unifying reads)"]
      direction LR
      Q[Query engine<br/>Trino / polystore] --> P[(Postgres)]
      Q --> M[(MySQL)]
      Q --> E[(Elastic)]
      Q --> S3[(S3)]
    end
    subgraph Unbundled["Unbundled DB (unifying writes)"]
      direction LR
      App2[App] --> L[(Event log)]
      L --> P2[(Postgres)]
      L --> E2[(Elastic)]
      L --> C2[(Cache)]
      L --> W2[(Warehouse)]
    end
```

"If we start from the premise that no single data model or storage format is suitable for all access patterns" — **federate to read across them, log to write across them.** See [[02-unbundling-the-database]].

### 3. Designing Applications Around Dataflow

Once writes flow through a log, application code naturally becomes a **derivation function**: a stateless, deterministic operator that consumes events and produces events. This is the separation of Church (application logic, lambda-calculus style) and state (managed log + store).

The most concrete payoff: **replace a synchronous RPC with a stream-table join**. Classic example from the chapter: you process purchases priced in one currency and paid in another.

- **Microservices way:** on every purchase, synchronously RPC to the exchange-rate service.
- **Dataflow way:** subscribe to the stream of exchange-rate updates ahead of time, keep the latest rate in a local store, and join it with purchase events locally.

"The fastest and most reliable network request is no network request at all."

```mermaid
flowchart LR
    Rates[Exchange rate<br/>stream] --> Local[(Local<br/>rate table)]
    Purchases[Purchase events] --> Join{Stream-table<br/>join}
    Local --> Join
    Join --> Priced[Priced purchase<br/>events]
```

Requires the message broker to guarantee **ordering + fault tolerance**. Also introduces **time-dependent joins**: if you reprocess history, which rate was active at that instant? (Event time vs. processing time.) See [[03-applications-around-dataflow]].

### 4. Write Path, Read Path, and Observing Derived State

Every derived dataset is a chosen point on a **write-path / read-path continuum**:

- **No index (grep-scan):** zero work on write, maximum work on read.
- **Full index:** moderate write work, fast read.
- **Cached common queries:** more write work (must update cache on relevant writes), near-zero read work for cached queries.
- **Per-user materialized view:** maximum write fan-out (celebrity problem!), trivial read.
- **Push to end-user device:** write path extends *all the way into the browser* via WebSockets / server-sent events; the UI is a subscriber.

```mermaid
flowchart LR
    Write[Write] -->|eager<br/>precompute| Derived[(Derived dataset)]
    Derived -->|lazy<br/>query| Read[Read]
    Derived -->|push via<br/>WebSocket| Client[End-user device<br/>UI as subscriber]
```

"After 500 pages, we have come full circle!" — the social-network home-timeline case study from Chapter 2 is exactly this boundary-shifting problem for celebrities vs. ordinary users.

Reads-can-be-events too: send read requests through the stream processor and you turn lookup into a stream-table join, enabling multishard queries and full causal provenance. See [[04-write-path-and-read-path]].

### 5. The End-to-End Argument and Idempotent Operations

**Saltzer, Reed, and Clark, 1984:**

> "The function in question can completely and correctly be implemented only with the knowledge and help of the application standing at the endpoints of the communication system."

TCP deduplicates packets — only within one connection. Database transactions are atomic — only if the client's reconnect after a timeout is handled. 2PC extends atomicity across shards — but not across the end-user's flaky cell network submitting a form twice.

**None of these is the endpoint that matters for "the user got charged exactly once."** Only the application logic, armed with a client-generated request ID carried through every hop, is.

The canonical pattern:

```sql
ALTER TABLE requests ADD UNIQUE (request_id);

BEGIN TRANSACTION;
INSERT INTO requests (request_id, from_acc, to_acc, amount)
VALUES ('uuid-0286...', 4321, 1234, 11.00);
UPDATE accounts SET balance = balance + 11.00 WHERE account_id = 1234;
UPDATE accounts SET balance = balance - 11.00 WHERE account_id = 4321;
COMMIT;
```

Retry the POST? Same `request_id`, uniqueness constraint rejects the second insert, transaction aborts, no double charge. The same argument applies to **integrity checksums** (bit-rot in app code is invisible to TCP/Ethernet CRCs) and to **encryption** (WiFi/TLS don't protect against a compromised server).

```mermaid
flowchart LR
    UI[End-user client<br/>generates request_id] -->|POST + id| App[App server]
    App -->|id forwarded| DB[(Database<br/>UNIQUE on request_id)]
    DB -->|duplicate -> abort| App
    App -->|response| UI
```

See [[05-end-to-end-argument-and-idempotence]].

### 6. Enforcing Constraints Without Coordination

Uniqueness requires consensus. In a distributed system, that usually means a single leader. The chapter's key insight: **route all potentially-conflicting writes to the same log shard** (by hash of the unique value), then a single-threaded stream processor deterministically orders them. That is consensus — per shard — with horizontal scale.

Multishard atomicity (the payment example: debit source, credit destination, collect fee) becomes:

1. Client generates a request ID; the initial request is appended atomically to one log shard (the source account's shard).
2. The source-account stream processor reads it, checks the balance, updates its local state, and emits events to the destination and fees shards (each tagged with the request ID).
3. Each downstream shard dedups on request ID and applies its half.

No atomic commit anywhere; integrity is preserved because (a) the single initial log append is atomic and (b) every downstream effect is derived deterministically from that append and idempotently applied.

```mermaid
flowchart LR
    Client -->|request_id + debit| Src[Source-account<br/>log shard]
    Src --> SPsrc[Stream processor<br/>source]
    SPsrc -->|event + id| Dst[Dest-account<br/>log shard]
    SPsrc -->|event + id| Fees[Fees-account<br/>log shard]
    Dst --> SPdst[Stream processor<br/>dest]
    Fees --> SPfee[Stream processor<br/>fees]
```

This works because the chapter **decouples timeliness from integrity**. Integrity ("no money disappears, no duplicate charge") is absolute; timeliness ("the payee sees the credit within 10ms") is a business choice. When apology is cheap (overbooked flights, overdrawn accounts, out-of-stock items), a **compensating transaction** is a better answer than paying the perf cost of coordination.

"You cannot reduce the number of apologies to zero, but you can aim to find the best trade-off for your needs." See [[06-coordination-avoiding-constraints]].

### 7. Trust, but Verify — Auditable Dataflow Systems

Every layer lies occasionally — disks flip bits, networks drop frames, MySQL has had uniqueness-constraint bugs, PostgreSQL's serializable mode has had write-skew bugs. If you believe your system is perfect because serializable transactions say so, you will not notice when it isn't.

Mature systems **continually re-derive state from the immutable event log and hash-check it**:

- HDFS and S3 run background scrubbers that read every replica and compare.
- Event sourcing gives free **provenance**: replay the log through the current derivation code and the output should match.
- **Merkle trees** give efficient proofs that a record is in a log without trusting the log server.
- **Certificate transparency** is a real-world deployment of single-leader append-only Merkle logs — no Byzantine consensus needed, just a public append-only hash chain.
- **Blockchains** are the Byzantine-fault-tolerant extreme: expensive, but no trusted leader needed.

The end-to-end argument, applied to **integrity checks**: only a check spanning the entire pipeline catches all corruption sources.

```mermaid
flowchart LR
    Events[(Immutable<br/>event log)] --> Derive1[Derive view A]
    Events --> Derive2[Derive view A<br/>redundant run]
    Derive1 --> HashA[Hash / Merkle root]
    Derive2 --> HashB[Hash / Merkle root]
    HashA --> Cmp{Compare<br/>periodically}
    HashB --> Cmp
    Cmp -->|mismatch| Alert[Alert + re-derive]
```

See [[07-trust-but-verify-auditability]].

## Trade-offs and Alternatives

| Axis | Option A | Option B | When A wins | When B wins |
|------|----------|----------|-------------|-------------|
| **Integration** | Distributed transactions (XA / 2PC) | Log-based derived data | Homogeneous stores, low shard count, need read-your-writes across all views | Heterogeneous tools, high fan-out, fault isolation matters, evolvability matters |
| **Composition** | Integrated database (all features in one engine) | Unbundled (small tools + event log) | A single engine meets all workloads; operational simplicity trumps breadth | No single tool fits all access patterns; different teams own different pieces |
| **Read composition** | Federated query engine (Trino, FDW) | Denormalized materialized views | Ad-hoc cross-store analytics; tolerant of query-time translation cost | Predictable, hot query patterns; cheap reads required |
| **Inter-service calls** | Synchronous REST / RPC | Async stream-table join (local state) | Rare calls, tight consistency, simple deployments | Hot path, availability-sensitive, natural event stream available (e.g., rates, prices) |
| **Correctness mechanism** | TCP + DB transactions | End-to-end request ID + idempotence | Single-hop, same process, same connection | Anything spanning browser → load balancer → service → DB |
| **Constraint enforcement** | Strict coordination (leader + linearizable) | Coordination-avoiding (sharded log + compensating transactions) | Apology is unacceptable (flight takeoff clearance, nuclear launch) | Apology is cheap (overbooking, overdraft, out-of-stock), scale / geo matters |
| **Write-path/read-path boundary** | Thin (grep-scan, no index) | Thick (per-user materialized view, push to device) | Writes dominate, reads are cold | Reads dominate, offline UX matters, real-time UI |
| **Auditing** | Periodic backups + hope | Event-sourced re-derivation + Merkle / hash chains | Low-stakes, internal data | Financial, regulated, or at a scale where "unlikely" corruption is frequent |
| **Trust model** | Trust replicas / CRC / TLS | Continuous end-to-end integrity checks | Resources tight, low corruption cost | Large scale, high consequences, or multi-org (certificate transparency, supply-chain) |

## Key Takeaways

1. **No single tool fits every access pattern.** Composition is the norm; the integration story dominates the architecture.
2. **Log-based derived data is the pragmatic replacement for distributed transactions** in heterogeneous environments — loose coupling, fault containment, and evolvability outweigh the cost of asynchrony.
3. **Unbundle the database.** Treat CDC + event log as the spine; each derived store (index, cache, view, warehouse, model) is a Unix-style tool hanging off it.
4. **Application code = derivation function.** Keep it stateless and deterministic; put state management in the log and the managed stores.
5. **The write path can reach the end-user device.** WebSockets + local state turn the UI into a subscriber to the system of record.
6. **Correctness is end-to-end.** TCP, DB transactions, and 2PC are useful probability reducers, not sufficient. Thread a client request ID through every hop and enforce idempotence at the endpoint.
7. **Separate timeliness from integrity.** Integrity is non-negotiable; timeliness is a cost/latency/availability trade. Coordination is expensive — reserve it for the constraints where apology fails.
8. **Trust, but verify.** Continuously re-derive and hash-check. Merkle trees and certificate-transparency-style logs are cheaper than Byzantine consensus and almost always enough.

## See Also

- [[01-data-integration-via-derived-data]] — why log-based derivations beat XA / 2PC for heterogeneous integration
- [[02-unbundling-the-database]] — federated (reads) vs. unbundled (writes); the meta-database of an organization
- [[03-applications-around-dataflow]] — stream-table join as RPC replacement; Church vs. state
- [[04-write-path-and-read-path]] — caches, indexes, and materialized views as boundary choices; push to the client
- [[05-end-to-end-argument-and-idempotence]] — Saltzer/Reed/Clark applied to duplicate suppression and integrity
- [[06-coordination-avoiding-constraints]] — sharded-log uniqueness, multishard transfer without atomic commit, timeliness vs. integrity
- [[07-trust-but-verify-auditability]] — event-sourced provenance, Merkle trees, certificate transparency, blockchains